In [3]:
#Bài 1
p_hoc_deu = 0.50
p_hoc_don = 0.30
p_it_hoc = 0.20
p_qua_deu = 0.95
p_qua_don = 0.70
p_qua_it = 0.30

p_qua = (
    p_hoc_deu * p_qua_deu
    + p_hoc_don * p_qua_don
    + p_it_hoc * p_qua_it
)

hau_nghiem_deu = p_hoc_deu * p_qua_deu / p_qua
hau_nghiem_don = p_hoc_don * p_qua_don / p_qua
hau_nghiem_it = p_it_hoc * p_qua_it / p_qua

print("XAC SUAT SINH VIEN THUOC TUNG NHOM KHI DA QUA MON")
print(f"Nhom hoc deu: {hau_nghiem_deu:.4f}")
print(f"Nhom hoc don: {hau_nghiem_don:.4f}")
print(f"Nhom it hoc:  {hau_nghiem_it:.4f}")
print(f"Tong xac suat: {hau_nghiem_deu + hau_nghiem_don + hau_nghiem_it:.4f}")

XAC SUAT SINH VIEN THUOC TUNG NHOM KHI DA QUA MON
Nhom hoc deu: 0.6376
Nhom hoc don: 0.2819
Nhom it hoc:  0.0805
Tong xac suat: 1.0000


Khi biết một sinh viên đã qua môn, xác suất sinh viên thuộc nhóm học đều là khoảng 63,76%, nhóm học dồn là 28,19% và nhóm ít học là 8,05%. Tổng ba xác suất bằng 1, phù hợp với yêu cầu đề bài.

In [4]:
#Bài 2
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]

mo_hinh = MultinomialNB(alpha=1.0)
mo_hinh.fit(X, y)
ten_tu = vec.get_feature_names_out()

hieu = mo_hinh.feature_log_prob_[1] - mo_hinh.feature_log_prob_[0]

bang = pd.DataFrame({
    "Tu": ten_tu,
    "Hieu_log": hieu
})

print("5 TU TO CAO TIN RAC MANH NHAT:")
print(bang.sort_values("Hieu_log", ascending=False)
      .head(5).to_string(index=False))

print("\n5 TU CHO THAY TIN THUONG MANH NHAT:")
print(bang.sort_values("Hieu_log", ascending=True)
      .head(5).to_string(index=False))

5 TU TO CAO TIN RAC MANH NHAT:
   Tu  Hieu_log
  mãi  3.203435
  tay  2.905600
 ngay  2.661837
 miễn  2.479516
trúng  2.374155

5 TU CHO THAY TIN THUONG MANH NHAT:
   Tu  Hieu_log
  lớp -2.612187
   em -2.561756
  bài -2.294553
  báo -2.274074
phòng -2.274074


Những từ có hiệu log dương lớn nghiêng về lớp tin rác; những từ có hiệu log âm nhỏ nhất nghiêng về lớp tin thường. Kết quả cụ thể được chương trình in ra dựa trên dữ liệu tin_nhan.csv

In [5]:
#Bài 3
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["noi_dung"],
    df["la_rac"],
    test_size=0.25,
    random_state=17,
    stratify=df["la_rac"]
)

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X_train = vec.fit_transform(X_train_text)
X_test = vec.transform(X_test_text)

cac_alpha = [0.001, 0.1, 1.0, 10.0, 100.0]

print("KET QUA SO SANH CAC GIA TRI ALPHA")
print("-" * 65)
print(f"{'Alpha':>10} {'Accuracy':>12} {'Precision':>12} {'Recall':>12}")

for alpha in cac_alpha:
    mo_hinh = MultinomialNB(alpha=alpha)
    mo_hinh.fit(X_train, y_train)

    y_pred = mo_hinh.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)

    print(f"{alpha:10.3f} {acc:12.4f} {pre:12.4f} {rec:12.4f}")

KET QUA SO SANH CAC GIA TRI ALPHA
-----------------------------------------------------------------
     Alpha     Accuracy    Precision       Recall
     0.001       0.9467       1.0000       0.8667
     0.100       0.9467       1.0000       0.8667
     1.000       0.9467       1.0000       0.8667
    10.000       0.9467       1.0000       0.8667
   100.000       0.9467       1.0000       0.8667


Khi alpha quá lớn, các xác suất xuất hiện của từ trong các lớp bị kéo lại gần nhau. Mô hình khó phân biệt từ đặc trưng của tin rác và tin thường, vì vậy khả năng phân loại có thể giảm.

In [6]:
#Bài 4
import math
from collections import Counter
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["noi_dung"].tolist(),
    df["la_rac"].tolist(),
    test_size=0.25,
    random_state=17,
    stratify=df["la_rac"]
)

def tach_tu(cau):
    return str(cau).split()

tu_dien = sorted({
    tu
    for cau in X_train_text
    for tu in tach_tu(cau)
})
V = len(tu_dien)

dem_tu = {0: Counter(), 1: Counter()}
tong_tu = {0: 0, 1: 0}
so_tin = {0: 0, 1: 0}

for cau, nhan in zip(X_train_text, y_train):
    so_tin[nhan] += 1
    for tu in tach_tu(cau):
        dem_tu[nhan][tu] += 1
        tong_tu[nhan] += 1

alpha = 1.0
log_xac_suat_tu = {}

for nhan in [0, 1]:
    log_xac_suat_tu[nhan] = {}
    for tu in tu_dien:
        p = (dem_tu[nhan][tu] + alpha) / (
            tong_tu[nhan] + alpha * V
        )
        log_xac_suat_tu[nhan][tu] = math.log(p)

def du_doan(cau):
    diem = {}

    for nhan in [0, 1]:
        diem[nhan] = math.log(
            so_tin[nhan] / sum(so_tin.values())
        )

        for tu in tach_tu(cau):
            if tu in tu_dien:
                diem[nhan] += log_xac_suat_tu[nhan][tu]

    return 1 if diem[1] > diem[0] else 0

y_pred_tu_viet = [du_doan(cau) for cau in X_test_text]

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X_train = vec.fit_transform(X_train_text)
X_test = vec.transform(X_test_text)

mo_hinh = MultinomialNB(alpha=1.0)
mo_hinh.fit(X_train, y_train)
y_pred_thu_vien = mo_hinh.predict(X_test)

acc_tu_viet = accuracy_score(y_test, y_pred_tu_viet)
acc_thu_vien = accuracy_score(y_test, y_pred_thu_vien)

print("KET QUA SO SANH")
print(f"Accuracy tu cai dat: {acc_tu_viet:.4f}")
print(f"Accuracy thu vien:   {acc_thu_vien:.4f}")
print(f"Do chenh lech:       {abs(acc_tu_viet - acc_thu_vien):.4f}")

if abs(acc_tu_viet - acc_thu_vien) < 0.05:
    print("Nhan xet: Hai mo hinh cho ket qua gan nhau.")
else:
    print("Nhan xet: Ket qua chenh lech, can kiem tra lai cach cai dat.")

KET QUA SO SANH
Accuracy tu cai dat: 0.9467
Accuracy thu vien:   0.9467
Do chenh lech:       0.0000
Nhan xet: Hai mo hinh cho ket qua gan nhau.


Hai mô hình sử dụng cùng dữ liệu huấn luyện, cùng tập kiểm tra và cùng hệ số làm trơn Laplace. Accuracy của hai mô hình dự kiến gần nhau. Nếu kết quả chênh lệch đáng kể, cần kiểm tra cách đếm từ, tính xác suất tiên nghiệm và tính log xác suất của từng từ.

In [7]:
#Bài 5
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, GaussianNB

df = pd.read_csv("data/sinh_vien.csv")

X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

nb_da_thuc = MultinomialNB()
nb_da_thuc.fit(X_train, y_train)
pred_da_thuc = nb_da_thuc.predict(X_test)
acc_da_thuc = accuracy_score(y_test, pred_da_thuc)

nb_gaussian = GaussianNB()
nb_gaussian.fit(X_train, y_train)
pred_gaussian = nb_gaussian.predict(X_test)
acc_gaussian = accuracy_score(y_test, pred_gaussian)

print("KET QUA SO SANH")
print(f"MultinomialNB Accuracy: {acc_da_thuc:.4f}")
print(f"GaussianNB Accuracy:    {acc_gaussian:.4f}")

KET QUA SO SANH
MultinomialNB Accuracy: 0.8000
GaussianNB Accuracy:    0.9000


Chương trình có thể chạy bình thường nhưng MultinomialNB không phù hợp về nguyên tắc vì mô hình này được thiết kế cho dữ liệu đếm. Số giờ ôn tập và điểm giữa kỳ là các đặc trưng số liên tục, nên GaussianNB phù hợp hơn. Accuracy cụ thể sẽ được in ra khi chạy với dữ liệu của bạn.

In [8]:

#Bài 6
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")

df_thuong = df[df["la_rac"] == 0]
df_rac = df[df["la_rac"] == 1].head(20)

df_moi = pd.concat([df_thuong, df_rac], ignore_index=True)

print("SO LUONG TIN NHAN")
print("Tin thuong:", len(df_thuong))
print("Tin rac:", len(df_rac))
print("Tong cong:", len(df_moi))
print()

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df_moi["noi_dung"],
    df_moi["la_rac"],
    test_size=0.25,
    random_state=17,
    stratify=df_moi["la_rac"]
)

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X_train = vec.fit_transform(X_train_text)
X_test = vec.transform(X_test_text)

mo_hinh = MultinomialNB(alpha=1.0)
mo_hinh.fit(X_train, y_train)

y_pred = mo_hinh.predict(X_test)

acc = accuracy_score(y_test, y_pred)
rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)

print("KET QUA DANH GIA")
print(f"Accuracy: {acc:.4f}")
print(f"Recall tin rac: {rec:.4f}")

SO LUONG TIN NHAN
Tin thuong: 180
Tin rac: 20
Tong cong: 200

KET QUA DANH GIA
Accuracy: 0.9800
Recall tin rac: 0.8000


Bộ dữ liệu mới có 180 tin thường và 20 tin rác, khiến lớp tin rác chỉ chiếm 10% tổng dữ liệu. Accuracy có thể gây hiểu nhầm vì mô hình có thể dự đoán thiên về lớp thường. Recall của lớp rác giúp đánh giá rõ hơn khả năng phát hiện tin rác thực tế.